<a href="https://www.kaggle.com/code/shravankumarpandey/next-word-prediction-using-lstm?scriptVersionId=324713911" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shravankumarpandey/large-scale-english-text-dataset/lstm_next_word_corpus.txt


In [16]:
with open("/kaggle/input/datasets/shravankumarpandey/large-scale-english-text-dataset/lstm_next_word_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Importing Libraries

In [17]:
import tensorflow 
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding

# Initialize Tokenizer

In [18]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])

In [19]:
tokenizer.word_index

{'and': 1,
 'to': 2,
 'in': 3,
 'many': 4,
 'that': 5,
 'being': 6,
 'people': 7,
 'new': 8,
 'influence': 9,
 'often': 10,
 'discussions': 11,
 'about': 12,
 'experts': 13,
 'emphasize': 14,
 'the': 15,
 'importance': 16,
 'of': 17,
 'observation': 18,
 'learning': 19,
 'adaptation': 20,
 'artificial': 21,
 'intelligence': 22,
 'technology': 23,
 'business': 24,
 'education': 25,
 'travel': 26,
 'daily': 27,
 'athletes': 28,
 'improve': 29,
 'performance': 30,
 'through': 31,
 'training': 32,
 'recovery': 33,
 'strategic': 34,
 'decision': 35,
 'making': 36,
 'economic': 37,
 'conditions': 38,
 'can': 39,
 'affect': 40,
 'employment': 41,
 'opportunities': 42,
 'investment': 43,
 'decisions': 44,
 'consumer': 45,
 'behavior': 46,
 'is': 47,
 'applied': 48,
 'solve': 49,
 'problems': 50,
 'healthcare': 51,
 'finance': 52,
 'researchers': 53,
 'study': 54,
 'complex': 55,
 'systems': 56,
 'understand': 57,
 'patterns': 58,
 'appear': 59,
 'nature': 60,
 'society': 61,
 'modern': 62,
 'c

In [20]:
l=len(tokenizer.word_index)
print(l)

161


In [21]:
for sentence in text.split("\n"):
    print(sentence)

In discussions about technology, many experts emphasize the importance of observation, learning, and adaptation. Creative thinking helps people adapt to changing circumstances and unexpected challenges. A successful business usually balances innovation, customer satisfaction, and long term planning. Daily routines may seem ordinary, yet they influence productivity and personal growth. People often learn new skills by practicing consistently and reflecting on their experiences. Travel exposes individuals to different cultures, languages, and perspectives. Modern technology continues to influence communication, transportation, and entertainment in many ways. Students benefit from curiosity, discipline, and access to reliable information sources. Researchers study complex systems to understand patterns that appear in nature and society.

In discussions about sports, many experts emphasize the importance of observation, learning, and adaptation. Historical events often shape political inst

In [22]:
input_subsequences = []

for sentence in text.split("\n"):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(tokenized_sentence)):
        input_subsequences.append(tokenized_sentence[:i+1])  

In [23]:
max_len=max([len(x) for x in input_subsequences])
print(max_len)

114


# Padding

In [24]:
padded_input_sequences=pad_sequences(input_subsequences,maxlen=max_len,padding="pre")
print(padded_input_sequences)

[[  0   0   0 ...   0   3  11]
 [  0   0   0 ...   3  11  12]
 [  0   0   0 ...  11  12  23]
 ...
 [  0   0   0 ... 152   2 153]
 [  0   0   0 ...   2 153 154]
 [  0   0   0 ... 153 154 155]]


In [25]:
X=padded_input_sequences[:,:-1]
y=padded_input_sequences[:,-1]

In [26]:
X.shape

(51550, 113)

In [27]:
y.shape

(51550,)

# One Hot Encoding

In [32]:
y=to_categorical(y,num_classes=l+1)

# Model Building

In [34]:
model= Sequential()
model.add(Embedding(l+1,100,input_length=max_len))
model.add(LSTM(150))
model.add(Dense(l+1,activation="softmax"))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1780652807.338785      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780652807.344712      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Model Compilation

In [35]:
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

# Model Fitting

In [36]:
model.fit(
    X,
    y,
    epochs=25
)

Epoch 1/25


I0000 00:00:1780652833.608652     180 cuda_dnn.cc:529] Loaded cuDNN version 91002


1611/1611 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - accuracy: 0.7900 - loss: 0.9928
Epoch 2/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9206 - loss: 0.2425
Epoch 3/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9206 - loss: 0.2357
Epoch 4/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9209 - loss: 0.2336
Epoch 5/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9205 - loss: 0.2321
Epoch 6/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9210 - loss: 0.2309
Epoch 7/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9206 - loss: 0.2302
Epoch 8/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9211 - loss: 0.2301
Epoch 9/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9214 - loss: 0.2297
Epoch 10/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9210 - loss: 0.2288
Epoch 11/25
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9214 - loss: 0.2284
Epoch 12/25
1611/1611 ━━━━━━━━

# Model Prediction

In [29]:
import numpy as np

def predict_next_word(text, num_words=10):
    for _ in range(num_words):

        token_text = tokenizer.texts_to_sequences([text])[0]

        padded_token = pad_sequences(
            [token_text],
            maxlen=max_len-1,
            padding="pre"
        )

        prediction = model.predict(padded_token, verbose=0)

        pos = np.argmax(prediction, axis=-1)[0]

        output_word = ""

        for word, index in tokenizer.word_index.items():
            if index == pos:
                output_word = word
                break

        text = text + " " + output_word

    return text

In [37]:
seed_text = "artificial intelligence is"
generated_text = predict_next_word(seed_text, num_words=10)

print(generated_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
artificial intelligence is many
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
artificial intelligence is many experts
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
artificial intelligence is many experts emphasize
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
artificial intelligence is many experts emphasize the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
artificial intelligence is many experts emphasize the importance
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
artificial intelligence is many experts emphasize the importance of
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
artificial intelligence is many experts emphasize the importance of observation
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
artificial intelligence is many experts emphasize the importance of observation learning
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
artificial intelligence is many experts emphasize the importance of observation learning and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
artificial intelligence is many experts emphasiz